<a href="https://colab.research.google.com/github/MarquiseRosier/pi05-run/blob/feature/marquise-transcoder-feature-inspection/notebooks/pi05_transcoder_counterfactual.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Pi0.5 Transcoder Counterfactual Probe

Standalone notebook for one study: **which transcoder features respond to a
single object's appearance, and does a traced circuit carry that response?**

It recolors one object at a frozen simulator state, renders the scene twice,
and pushes both observations through the policy with the same flow-matching
noise. Physics is never stepped between the renders, so any activation
difference is attributable to that object's pixels.

## What each stage decides

| Stage | Question | Decided on |
|---|---|---|
| Probe | H1: do features respond to the task object more than to an identical placebo, once footprint is accounted for? | `decision_metrics.json` → `h1` |
| Probe | H2: does that selectivity follow the language, or the object's position? | `decision_metrics.json` → `h2` |
| Report | Which feature to trace | `nominated_targets.json` (one rule, applied once) |
| Trace + Validate | H3: do the traced parents carry the perturbed property? | `circuit/counterfactual_validation/validation.json` → `h3_verdict` |

Every decision rule is written next to its verdict in those files.

## Why this is separate from the main notebook

The full simulation notebook mounts Drive caches, downloads the LIBERO dataset
and runs closed-loop evals. None of that is needed for the probe. It uses
`make_env`, not `make_dataset`, so there is **no dataset download** for the
probe stage, and it runs forward passes rather than rollouts. Only the circuit
trace needs dataset rows.

Drive is mounted for one reason only: to read the transcoder checkpoint.

## Order

Run the cells top to bottom. Nothing needs to be filled in: the probe reads the
task's BDDL and perturbs its `obj_of_interest`, using another instance of the
same object type as a matched control.

1. Controls
2. Runtime check and Drive mount
3. Install
4. Clone repo
5. HF token and LIBERO assets
6. Resolve checkpoint
7. Run the probe (H1, H2, nomination)
8. Trace and validate the circuit (H3)


In [ ]:
# @title Controls

# Repo.
REPO_URL = "https://github.com/MarquiseRosier/pi05-run.git"  # @param {type:"string"}
REPO_BRANCH = "feature/marquise-transcoder-feature-inspection"  # @param {type:"string"}

# Drive is used only to read the transcoder checkpoint.
DRIVE_ROOT = "/content/drive/MyDrive/groot-run-shared-programmer908"  # @param {type:"string"}
TRANSCODER_DRIVE_PATH = "transcoders/pi05_libero/allframes_80-10-10_epoch1_b8_exp16_latest_lambda1e-4/step_027233.pt"  # @param {type:"string"}

# Scene.
POLICY_PATH = "lerobot/pi05_libero_finetuned"  # @param {type:"string"}
SUITE = "libero_spatial"  # @param ["libero_spatial", "libero_object", "libero_goal", "libero_10"]
TASK_ID = 0  # @param {type:"integer"}
SEED = 1000  # @param {type:"integer"}

# Perturbation. Both default to the task's own objects: TARGET becomes the
# BDDL's first obj_of_interest, PLACEBO_TARGET another instance of the same
# object type. Leave them blank unless you want to override.
TARGET = ""  # @param {type:"string"}
PLACEBO_TARGET = ""  # @param {type:"string"}
LIST_OBJECTS_ONLY = False  # @param {type:"boolean"}
PERTURBATION = "blend"  # @param ["blend", "set", "hue"]
COLOR = "1.0,0.2,0.1"  # @param {type:"string"}
DOSE = "0.5,1.0"  # @param {type:"string"}

# Replication. Cells = STATES x NOISE_SAMPLES x doses under the task prompt;
# these are the replicates behind the error bound on H1. Noise draws are seeded
# from SEED, so the run is reproducible bit for bit.
STATES = 2  # @param {type:"integer"}
STATE_STRIDE = 5  # @param {type:"integer"}
NOISE_SAMPLES = 2  # @param {type:"integer"}

# Nomination rule for the trace target (applied once, by the report, and read
# by the trace cell). A candidate must fire in at least MIN_TRACE_CONSISTENCY
# of the task prompt's cells, sit at layer >= MIN_TRACE_LAYER so it has parents
# to find, and respond at most 1/MIN_SELECTIVITY as much to the placebo.
MIN_TRACE_CONSISTENCY = 0.25  # @param {type:"number"}
MIN_TRACE_LAYER = 4  # @param {type:"integer"}
MIN_SELECTIVITY = 2.0  # @param {type:"number"}

# Circuit tracing. The tracer needs feature-discovery artifacts, which come
# from the LIBERO dataset rather than the live env, so this stage downloads the
# dataset. Scoped small: we only need top-K for the nominated feature.
RUN_CIRCUIT_TRACE = True  # @param {type:"boolean"}
DISCOVERY_EPISODES = "0,1,2,3,4"  # @param {type:"string"}
DISCOVERY_MAX_BATCHES = 40  # @param {type:"integer"}
TRACE_PARENTS_PER_NODE = 2  # @param {type:"integer"}
TRACE_MAX_DEPTH = 3  # @param {type:"integer"}
TRACE_MAX_NODES = 60  # @param {type:"integer"}
VALIDATION_RANDOM_DRAWS = 2000  # @param {type:"integer"}

MIN_GPU_MEMORY_GB = 20  # @param {type:"integer"}

print("Suite:", SUITE, "| task:", TASK_ID, "| seed:", SEED)
print("Cells per condition:", STATES * NOISE_SAMPLES * len([d for d in DOSE.split(",") if d.strip()]))
print("Target:", repr(TARGET) or "(discovery pass)")


In [ ]:
# @title Validate Runtime And Mount Drive

import os
import subprocess
import time
from pathlib import Path

from google.colab import drive


def gpu_info():
    try:
        out = subprocess.run(
            ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"],
            capture_output=True, text=True, check=True,
        ).stdout.strip().splitlines()[0]
        name, mem = [part.strip() for part in out.split(",")]
        return name, int(mem) // 1024
    except Exception:
        return None, 0


gpu_name, gpu_mem_gb = gpu_info()
if gpu_name is None:
    raise RuntimeError("No GPU. Runtime > Change runtime type > L4.")
print(f"GPU: {gpu_name} (~{gpu_mem_gb} GiB)")
if gpu_mem_gb < int(MIN_GPU_MEMORY_GB):
    raise RuntimeError(
        f"{gpu_name} has ~{gpu_mem_gb} GiB; Pi0.5 needs >= {MIN_GPU_MEMORY_GB} GiB. Use an L4 or A100."
    )


def mount_drive_with_retry(mountpoint: str = "/content/drive", attempts: int = 3) -> None:
    """Mount Drive, retrying because `mount failed` is usually transient."""
    if Path(mountpoint, "MyDrive").exists():
        print(f"Drive already mounted at {mountpoint}", flush=True)
        return
    last_error = None
    for attempt in range(1, attempts + 1):
        try:
            drive.mount(mountpoint, force_remount=attempt > 1)
            print(f"Drive mounted at {mountpoint} (attempt {attempt})", flush=True)
            return
        except Exception as exc:
            last_error = exc
            print(f"Drive mount attempt {attempt}/{attempts} failed: {exc}", flush=True)
            if attempt < attempts:
                delay = 5 * attempt
                print(f"  retrying in {delay}s", flush=True)
                time.sleep(delay)
    raise RuntimeError(
        f"Could not mount Google Drive after {attempts} attempts (last error: {last_error}).\n"
        "Drive is needed here only to read the transcoder checkpoint.\n"
        "Most likely causes, in order:\n"
        "  1. Other Colab sessions hold Drive mounts. Runtime > Manage sessions, terminate "
        "the ones you are not using, then Runtime > Restart session and rerun.\n"
        "  2. The authorization popup was blocked. Allow popups and third-party cookies.\n"
        "  3. A transient Drive outage. Restart the runtime and retry in a few minutes."
    ) from last_error


mount_drive_with_retry()
DRIVE_ROOT = Path(DRIVE_ROOT)
print("Drive root:", DRIVE_ROOT, "exists:", DRIVE_ROOT.exists())


In [ ]:
# @title Install Runtime

import os
import subprocess
from pathlib import Path

VENV = Path("/content/lerobot-venv")
PYTHON = VENV / "bin/python"
UV_BIN_DIR = Path("/content/uv-bin")
UV = str(UV_BIN_DIR / "uv")


def run(cmd, *, env=None):
    cmd = list(map(str, cmd))
    print("$", " ".join(cmd), flush=True)
    return subprocess.run(cmd, env=env, check=True)


if PYTHON.exists():
    print("venv already present; skipping install. Delete /content/lerobot-venv to force a rebuild.")
else:
    apt_packages = [
        "build-essential", "cmake", "curl", "ffmpeg", "git", "pkg-config",
        "libegl1", "libgl1", "libglib2.0-0", "libglvnd0", "libglx0", "libopengl0",
        "libosmesa6-dev", "libsm6", "libxext6", "libxrender1",
    ]
    apt_env = os.environ.copy()
    apt_env["DEBIAN_FRONTEND"] = "noninteractive"
    run(["apt-get", "update", "-qq"], env=apt_env)
    run(["apt-get", "install", "-y", "-qq", *apt_packages], env=apt_env)

    UV_BIN_DIR.mkdir(parents=True, exist_ok=True)
    run(["curl", "-LsSf", "https://astral.sh/uv/install.sh", "-o", "/tmp/install-uv.sh"])
    uv_env = os.environ.copy()
    uv_env["UV_INSTALL_DIR"] = str(UV_BIN_DIR)
    run(["sh", "/tmp/install-uv.sh"], env=uv_env)
    os.environ["PATH"] = f"{UV_BIN_DIR}:" + os.environ["PATH"]

    run([UV, "python", "install", "3.12"])
    run([UV, "venv", "--clear", str(VENV), "--python", "3.12"])
    # `evaluation` pulls the LIBERO env; no dataset extras are needed for this probe.
    run([
        UV, "pip", "install", "--python", str(PYTHON), "--torch-backend", "cu128",
        "lerobot[evaluation,libero,pi]", "hf-transfer", "opencv-python", "numpy",
    ])

os.environ["PATH"] = f"{VENV / 'bin'}:{UV_BIN_DIR}:" + os.environ["PATH"]
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
os.environ["MUJOCO_GL"] = "egl"
os.environ["PYOPENGL_PLATFORM"] = "egl"
run([str(PYTHON), "-c",
     "import torch; print('torch', torch.__version__, '| cuda', torch.cuda.is_available())"])


In [ ]:
# @title Clone Or Update Repo

import subprocess
from pathlib import Path

LOCAL_REPO = Path("/content/pi05-run")
if LOCAL_REPO.exists():
    subprocess.run(["git", "-C", str(LOCAL_REPO), "fetch", "origin", REPO_BRANCH], check=True)
    subprocess.run(["git", "-C", str(LOCAL_REPO), "checkout", REPO_BRANCH], check=True)
    subprocess.run(["git", "-C", str(LOCAL_REPO), "reset", "--hard", f"origin/{REPO_BRANCH}"], check=True)
else:
    subprocess.run(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(LOCAL_REPO)], check=True)
subprocess.run(["git", "-C", str(LOCAL_REPO), "log", "-1", "--oneline"], check=True)


In [ ]:
# @title HF Token And LIBERO Assets

import os
from pathlib import Path

token = os.environ.get("HF_TOKEN", "")
if not token:
    try:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN") or ""
    except Exception:
        token = ""
if not token:
    secret_file = DRIVE_ROOT / "secrets/HF_TOKEN.txt"
    if secret_file.exists():
        token = secret_file.read_text().strip()
if not token:
    raise RuntimeError(
        "No Hugging Face token. Add HF_TOKEN as a Colab secret (key icon in the sidebar), "
        f"or place it at {DRIVE_ROOT / 'secrets/HF_TOKEN.txt'}. "
        f"{POLICY_PATH} is gated, so the download needs it."
    )
os.environ["HF_TOKEN"] = token
print("HF token loaded.")

# LIBERO ships without its scene assets; fetch them into the installed package.
assets_code = r"""
import shutil, site, os
from pathlib import Path
from huggingface_hub import snapshot_download

roots = [Path(p) for p in site.getsitepackages()]
user_site = site.getusersitepackages()
if user_site:
    roots.append(Path(user_site))
libero_root = next((r / "libero" / "libero" for r in roots if (r / "libero" / "libero").exists()), None)
if libero_root is None:
    raise RuntimeError("Installed LIBERO package not found")
assets_dir = libero_root / "assets"
required = assets_dir / "scenes" / "libero_tabletop_base_style.xml"
if required.exists():
    print("LIBERO assets already present:", required)
else:
    snap = Path(snapshot_download(repo_id="lerobot/libero-assets", repo_type="dataset",
                                  token=os.environ.get("HF_TOKEN") or None))
    assets_dir.mkdir(parents=True, exist_ok=True)
    for child in snap.iterdir():
        if child.name == ".gitattributes":
            continue
        target = assets_dir / child.name
        if child.is_dir():
            shutil.copytree(child, target, dirs_exist_ok=True)
        else:
            shutil.copy2(child, target)
    if not required.exists():
        raise FileNotFoundError(f"LIBERO asset install failed; missing {required}")
    print("LIBERO assets installed:", required)
"""
run([str(PYTHON), "-c", assets_code])


In [ ]:
# @title Resolve Transcoder Checkpoint

import shutil
import time
from pathlib import Path

candidate = Path(TRANSCODER_DRIVE_PATH)
if not candidate.is_absolute():
    candidate = DRIVE_ROOT / candidate
if not candidate.exists():
    raise FileNotFoundError(
        f"Transcoder checkpoint not found: {candidate}\n"
        "Check TRANSCODER_DRIVE_PATH in Controls against the shared Drive folder."
    )

# Copy off Drive once. Reading a multi-GiB checkpoint repeatedly over the Drive
# FUSE mount is far slower than a local read, and it is the checkpoint load that
# the probe does on every run.
LOCAL_CHECKPOINT = Path("/content/checkpoints") / candidate.name
LOCAL_CHECKPOINT.parent.mkdir(parents=True, exist_ok=True)
if LOCAL_CHECKPOINT.exists() and LOCAL_CHECKPOINT.stat().st_size == candidate.stat().st_size:
    print("Local copy already present:", LOCAL_CHECKPOINT)
else:
    print(f"Copying {candidate.stat().st_size / 1024**3:.2f} GiB off Drive...", flush=True)
    started = time.time()
    shutil.copy2(candidate, LOCAL_CHECKPOINT)
    print(f"  done in {time.time() - started:.0f}s", flush=True)

TRANSCODER_CHECKPOINT = str(LOCAL_CHECKPOINT)
print("Checkpoint:", TRANSCODER_CHECKPOINT)
print("Size:", f"{LOCAL_CHECKPOINT.stat().st_size / 1024**3:.2f} GiB")


## Run The Probe

Just run it. The probe reads the task's BDDL and picks its own objects:

* **target** -- the task's first `obj_of_interest`. For `libero_spatial` task 0
  that is `akita_black_bowl_1`, the bowl the prompt refers to.
* **placebo** -- another instance of the same object type, here
  `akita_black_bowl_2`. Same mesh, same colour, same size, differing only in
  position and task relevance, which makes it a tightly matched control.

What to read in the output, in order:

1. `All keys loaded successfully!` -- the policy load. The run aborts otherwise.
2. `re-render liveness` -- the recolour moved pixels and the revert restored them.
3. `null control ... latent L2 mean=0` -- the nondeterminism floor. Any other
   value invalidates the paired comparison and the H1 verdict says so.
4. `--- decision metrics ---` -- the H1 and H2 tables, verdicts and rules.
5. `Nominated targets` -- the one rule for what gets traced.

Override `TARGET` / `PLACEBO_TARGET` in Controls only if you want something
else, or tick `LIST_OBJECTS_ONLY` to just inspect the scene.


In [ ]:
# @title Run Counterfactual Probe

import json
import os
import subprocess
import sys
import time
from pathlib import Path
from IPython.display import display, HTML, Markdown, Image as IPyImage

out_dir = LOCAL_REPO / "outputs/probes/counterfactual" / time.strftime("%Y%m%d-%H%M%S", time.gmtime())
out_dir.mkdir(parents=True, exist_ok=True)

cmd = [
    str(PYTHON), "-u", "scripts/probe_pi05_transcoder_counterfactual.py",
    "--policy-path", POLICY_PATH,
    "--output-dir", str(out_dir),
    "--suite", SUITE,
    "--task-id", str(TASK_ID),
    "--seed", str(SEED),
    "--device", "cuda",
    "--policy-dtype", "bfloat16",
]

target = TARGET.strip()
if LIST_OBJECTS_ONLY:
    cmd.append("--list-objects")
else:
    cmd += [
        "--checkpoint", TRANSCODER_CHECKPOINT,
        "--perturbation", PERTURBATION,
        "--color", COLOR,
        "--dose", DOSE,
        "--states", str(STATES),
        "--state-stride", str(STATE_STRIDE),
        "--noise-samples", str(NOISE_SAMPLES),
        "--noise-seed", str(SEED),
    ]
    # Omitted flags let the probe pick the task's own objects.
    if target:
        cmd += ["--target", target]
    if PLACEBO_TARGET.strip():
        cmd += ["--placebo-target", PLACEBO_TARGET.strip()]

log_path = out_dir / "probe.log"
print("$", " ".join(cmd), flush=True)
env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"
# Colab's inline matplotlib backend is only valid inside the kernel process;
# inheriting it breaks LIBERO's import. The script guards this too.
env["MPLBACKEND"] = "Agg"
env["MUJOCO_GL"] = "egl"
env["PYOPENGL_PLATFORM"] = "egl"
env["PYTHONPATH"] = str(LOCAL_REPO / "src") + (
    os.pathsep + env["PYTHONPATH"] if env.get("PYTHONPATH") else ""
)


def emit(chunk: bytes) -> None:
    """Write through to the cell output.

    Colab's sys.stdout is an ipykernel OutStream with no .buffer, so decode
    rather than assuming a binary stream exists.
    """
    stream = getattr(sys.stdout, "buffer", None)
    if stream is not None:
        stream.write(chunk)
    else:
        sys.stdout.write(chunk.decode("utf-8", errors="replace"))
    sys.stdout.flush()


with log_path.open("wb") as log_handle:
    process = subprocess.Popen(cmd, cwd=LOCAL_REPO, env=env,
                               stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=0)
    try:
        while True:
            chunk = process.stdout.read(4096)
            if not chunk:
                break
            log_handle.write(chunk); log_handle.flush()
            emit(chunk)
        rc = process.wait()
    except BaseException:
        # Never leave the probe running headless if the cell is interrupted.
        process.kill()
        process.wait()
        raise
if rc != 0:
    tail = log_path.read_text(errors="replace").splitlines()[-40:]
    print(f"\n--- last {len(tail)} log lines ({log_path}) ---", flush=True)
    for line in tail:
        print(line[-500:], flush=True)
    raise subprocess.CalledProcessError(rc, cmd)

if LIST_OBJECTS_ONLY:
    display(Markdown(
        "### Objects listed\n\nSet `LIST_OBJECTS_ONLY = False` to run the measurement. "
        "Override `TARGET` / `PLACEBO_TARGET` only if you do not want the task's own objects."
    ))
else:
    shapes_path = out_dir / "observation_shapes.json"
    if shapes_path.exists():
        shapes = json.loads(shapes_path.read_text())
        display(Markdown("### Observation Shapes Sent To The Policy"))
        display(HTML(
            "<table><tr><th>key</th><th>shape</th><th>dtype</th></tr>"
            + "".join(f"<tr><td>{k}</td><td>{v['shape']}</td><td>{v['dtype']}</td></tr>"
                      for k, v in sorted(shapes.get("tensor_shapes", {}).items()))
            + f"<tr><td>noise</td><td>{shapes.get('noise_shape')}</td>"
              f"<td>{shapes.get('noise_dtype')}</td></tr></table>"
        ))
        print("task:", repr(shapes.get("task")))

    decision_path = out_dir / "decision_metrics.json"
    if decision_path.exists():
        decision = json.loads(decision_path.read_text())
        h1, h2 = decision["h1"], decision["h2"]

        def _f(v, spec=".4g"):
            return "n/a" if v is None else format(float(v), spec)

        pooled, spread, ci = h1["pooled"], h1["spread"], h1["bootstrap_ci_95"]
        display(Markdown(f"### H1 -- object selectivity (task prompt, {len(h1['cells'])} cells)"))
        display(HTML(
            "<table><tr><th>quantity</th><th>target</th><th>placebo</th><th>ratio</th></tr>"
            f"<tr><td>D, layer response (latent L2)</td><td>{_f(pooled['D_target'])}</td><td>{_f(pooled['D_placebo'])}</td><td>{_f(pooled['sel_raw'])}</td></tr>"
            f"<tr><td>S, changed-pixel fraction</td><td>{_f(pooled['S_px_target'])}</td><td>{_f(pooled['S_px_placebo'])}</td><td>{_f(pooled['S_px_ratio'])}</td></tr>"
            f"<tr><td>S, image-difference norm</td><td>{_f(pooled['S_l2_target'])}</td><td>{_f(pooled['S_l2_placebo'])}</td><td>{_f(pooled['S_l2_ratio'])}</td></tr>"
            f"<tr><td>action relative L2</td><td>{_f(pooled['A_target'])}</td><td>{_f(pooled['A_placebo'])}</td><td>{_f(pooled['action_sel'])}</td></tr>"
            "</table>"
        ))
        display(HTML(
            "<table><tr><th>selectivity</th><th>pooled</th><th>cells mean ± sd</th><th>min .. max</th><th>95% CI</th></tr>"
            + "".join(
                f"<tr><td>{label}</td><td>{_f(pooled[k])}</td><td>{_f(spread[k]['mean'])} ± {_f(spread[k]['sd'])}</td>"
                f"<td>{_f(spread[k]['min'])} .. {_f(spread[k]['max'])}</td><td>[{_f(ci[k]['low'])}, {_f(ci[k]['high'])}]</td></tr>"
                for k, label in (("sel_raw", "raw"), ("sel_adj_px", "adjusted, |P|"), ("sel_adj_l2", "adjusted, ||dI||"))
            )
            + "</table>"
        ))
        nf = h1["null_floor"]
        print(f"null floor D: mean={_f(nf['mean'])} max={_f(nf['max'])} -> {'near zero' if h1['null_floor_near_zero'] else 'NOT near zero'}")
        for kind, dr in h1["dose_response"].items():
            print(f"dose response [{kind}]: elasticity={_f(dr['elasticity_mean'])} monotone={dr['monotone_in_dose']}  "
                  + "  ".join(f"d={r['dose']:g}: D={_f(r['D'])}" for r in dr["rows"]))
        display(Markdown(f"**H1 verdict: {h1['verdict']}**  \n<small>rule: {h1['decision_rule']}</small>"))

        g = h2["grounding"]
        display(Markdown("### H2 -- referent versus position (prompt swap over identical pixels)"))
        display(HTML(
            "<table><tr><th>prompt</th><th>D target</th><th>D placebo</th><th>selectivity</th><th>cells</th></tr>"
            + "".join(
                f"<tr><td>{p}</td><td>{_f(r['D_target'])}</td><td>{_f(r['D_placebo'])}</td><td>{_f(r['sel_raw'])}</td><td>{r['n_cells']}</td></tr>"
                for p, r in h2["per_prompt"].items()
            )
            + "</table>"
        ))
        print(f"prompt grounding g: mean={_f(g['mean'])} min={_f(g['min'])} max={_f(g['max'])} (n={g['n']}, threshold {h2['grounding_threshold']}); "
              f"perturbation action effect {_f(h2['perturbation_action_effect'])}")
        display(Markdown(f"**H2 verdict: {h2['verdict']}**  \n<small>rule: {h2['decision_rule']}</small>"))

    summary_path = out_dir / "counterfactual_summary.json"
    if summary_path.exists():
        summary = json.loads(summary_path.read_text())
        display(Markdown("### Counterfactual Activation Response (every measurement)"))
        display(HTML(
            "<table><tr><th>state</th><th>noise</th><th>prompt</th><th>kind</th><th>target</th><th>dose</th>"
            "<th>changed pixels</th><th>latent L2</th><th>action rel L2</th></tr>"
            + "".join(
                "<tr><td>{s}</td><td>{n}</td><td>{p}</td><td>{k}</td><td>{t}</td><td>{d}</td><td>{px:.5f}</td>"
                "<td>{lat}</td><td>{act:.5g}</td></tr>".format(
                    s=r["state_index"], n=r.get("noise_index", 0), p=r.get("prompt", "task"),
                    k=r["kind"], t=r["target"], d=r["dose"],
                    px=r["pixel"]["changed_pixel_fraction"],
                    lat=("n/a" if r["latent"].get("l2_delta_mean") is None
                         else "{:.5g}".format(r["latent"]["l2_delta_mean"])),
                    act=r["action_relative_l2"])
                for r in summary.get("measurements", []))
            + "</table>"
        ))
        print("Read every row against the 'null' rows: that is the nondeterminism floor.")

    for png in sorted((out_dir / "images").glob("*.png"))[:12]:
        display(Markdown(f"**{png.name}**"))
        display(IPyImage(filename=str(png)))

    # Rank features by selectivity: responds to the task object, not to the
    # visually matched control.
    report_cmd = [str(PYTHON), "-u", "scripts/report_pi05_counterfactual_features.py",
                  str(out_dir), "--top", "30",
                  "--min-consistency", str(MIN_TRACE_CONSISTENCY),
                  "--min-trace-layer", str(MIN_TRACE_LAYER),
                  "--min-selectivity", str(MIN_SELECTIVITY)]
    print("\n$", " ".join(report_cmd), flush=True)
    report = subprocess.run(report_cmd, cwd=LOCAL_REPO, env=env, capture_output=True, text=True)
    print(report.stdout)
    if report.returncode != 0:
        print(report.stderr[-2000:])

print("\nArtifacts:", out_dir)


## Trace And Validate The Circuit

The report nominated a target feature by *controlled perturbation
selectivity*, under one rule written to `nominated_targets.json`. This cell
reads that file: there is no second selection step here.

Feature discovery nominates by *activation statistics*; both feed the same
tracer, which walks backward from the target through the replacement model
using gradient attribution. Discovery needs the LIBERO dataset, because the
tracer runs its attribution on the observations where the target feature
actually fires, and those come from dataset rows rather than the live env.

Then the traced circuit is scored against the probe's full delta store. Read
the output in this order:

1. `exercised parents` -- how much of the circuit this scene activates at all.
   Below half, the verdict is inconclusive by rule.
2. The `target` row -- enrichment of the parents over matched random features
   at the same layer and flow time, with its Monte-Carlo p. The target node is
   excluded: it was picked *for* responding.
3. The `placebo` row and the circuit's own selectivity.
4. `H3 verdict`.

Worth watching: the target is nominated from *env* renders but traced on
*dataset* observations. If the feature does not fire in the discovery pool, the
trace has nothing to attribute and the cell says so.


In [ ]:
# @title Run Circuit Trace On The Nominated Feature

import csv
import json
import os
import subprocess
import sys
import time
from pathlib import Path
from IPython.display import display, HTML, Markdown

if not RUN_CIRCUIT_TRACE:
    print("RUN_CIRCUIT_TRACE is off; skipping.")
else:
    nominated_path = out_dir / "nominated_targets.json"
    if not nominated_path.exists():
        raise RuntimeError(f"No nominated_targets.json in {out_dir}. Run the probe cell first.")
    nomination = json.loads(nominated_path.read_text())
    criteria, nominated = nomination["criteria"], nomination["nominated"]
    if not nominated:
        raise RuntimeError(
            "The report nominated nothing under its rule "
            f"(rejections: {criteria.get('rejected')}). Widen the probe (STATES, NOISE_SAMPLES, DOSE) "
            "or relax MIN_TRACE_CONSISTENCY / MIN_TRACE_LAYER / MIN_SELECTIVITY deliberately, and say so "
            "in the write-up."
        )
    chosen = nominated[0]
    target_key = chosen["feature_key"]
    exact = chosen.get("selectivity_exact")
    print(
        f"nominated target: {target_key}  z={float(chosen['target_z']):+.2f}  "
        f"top-K sel {float(chosen['selectivity']):.1f}x  "
        + (f"exact sel {float(exact):.1f}x  " if exact not in (None, "", "None") else "exact placebo 0  ")
        + f"cells {chosen['target_cells']}/{chosen['layer_cells']} ({float(chosen['consistency']):.0%})",
        flush=True,
    )
    print(f"rule: prompt={criteria['prompt']}, consistency>={criteria['min_consistency']}, "
          f"layer>={criteria['min_trace_layer']}, {criteria['placebo_rule']}; rejected {criteria['rejected']}",
          flush=True)

    stamp = time.strftime("%Y%m%d-%H%M%S", time.gmtime())
    feature_dir = LOCAL_REPO / "outputs/features/pi05_libero" / f"counterfactual-{stamp}"
    trace_dir = out_dir / "circuit"

    env = os.environ.copy()
    env["PYTHONUNBUFFERED"] = "1"
    env["MPLBACKEND"] = "Agg"
    env["MUJOCO_GL"] = "egl"
    env["PYOPENGL_PLATFORM"] = "egl"
    # collect_ and trace_ import pi05_mi as an installed package; only the probe
    # puts src/ on sys.path itself. This notebook never pip-installs the repo,
    # so hand them the path explicitly.
    src_path = str(LOCAL_REPO / "src")
    env["PYTHONPATH"] = src_path + (os.pathsep + env["PYTHONPATH"] if env.get("PYTHONPATH") else "")

    def stream(label, cmd, log_path):
        print(f"\n== {label} ==\n$ {' '.join(map(str, cmd))}", flush=True)
        log_path.parent.mkdir(parents=True, exist_ok=True)
        with log_path.open("wb") as handle:
            proc = subprocess.Popen([str(c) for c in cmd], cwd=LOCAL_REPO, env=env,
                                    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=0)
            try:
                while True:
                    chunk = proc.stdout.read(4096)
                    if not chunk:
                        break
                    handle.write(chunk); handle.flush()
                    stream_out = getattr(sys.stdout, "buffer", None)
                    if stream_out is not None:
                        stream_out.write(chunk)
                    else:
                        sys.stdout.write(chunk.decode("utf-8", errors="replace"))
                    sys.stdout.flush()
                return proc.wait()
            except BaseException:
                proc.kill(); proc.wait(); raise

    # 1. Feature discovery, scoped small: the tracer only needs top-K for this target.
    rc = stream(
        "Feature discovery (for the tracer's top-K observations)",
        [PYTHON, "-u", "scripts/collect_pi05_transcoder_features.py",
         "--checkpoint", TRANSCODER_CHECKPOINT, "--output-dir", feature_dir,
         "--episodes", DISCOVERY_EPISODES, "--max-batches", str(DISCOVERY_MAX_BATCHES),
         "--device", "cuda", "--policy-dtype", "bfloat16"],
        feature_dir / "discovery.log",
    )
    if rc != 0:
        raise RuntimeError(f"feature discovery exited {rc}; see {feature_dir / 'discovery.log'}")

    # 2. Trace the circuit upstream of the nominated feature.
    rc = stream(
        f"Circuit trace from {target_key}",
        [PYTHON, "-u", "scripts/trace_pi05_transcoder_circuit.py",
         "--checkpoint", TRANSCODER_CHECKPOINT, "--feature-dir", feature_dir,
         "--output-dir", trace_dir, "--target", target_key,
         "--parents-per-node", str(TRACE_PARENTS_PER_NODE),
         "--max-depth", str(TRACE_MAX_DEPTH), "--max-nodes", str(TRACE_MAX_NODES),
         "--device", "cuda", "--policy-dtype", "bfloat16"],
        trace_dir / "trace.log",
    )
    if rc != 0:
        print(
            f"\nTrace exited {rc}. The usual cause is that {target_key} never fires in the "
            f"discovery pool ({DISCOVERY_EPISODES}), so there are no observations to attribute "
            "through. Widen DISCOVERY_EPISODES, or trace a target nominated by discovery itself.",
            flush=True,
        )
    else:
        for name in ("circuit.html", "trace.html", "circuit_trace.html"):
            candidate = trace_dir / name
            if candidate.exists():
                display(Markdown(f"### Circuit trace: `{target_key}`"))
                display(HTML(candidate.read_text()))
                break
        print("\nTrace artifacts:", trace_dir, flush=True)

        # Validate the traced circuit against the controlled perturbation. The
        # probe knows what changed in the input, so it can say whether the
        # traced parents actually carry that information or not.
        rc = stream(
            "Validate the circuit against the counterfactual",
            [PYTHON, "-u", "scripts/validate_circuit_with_counterfactual.py",
             trace_dir, out_dir, "--random-draws", str(VALIDATION_RANDOM_DRAWS)],
            trace_dir / "validation.log",
        )
        if rc != 0:
            print(f"validation exited {rc}; see {trace_dir / 'validation.log'}", flush=True)
        else:
            validation_path = trace_dir / "counterfactual_validation" / "validation.json"
            if validation_path.exists():
                v = json.loads(validation_path.read_text())
                display(Markdown("### H3 -- circuit corroboration"))
                def _h3_row(name, stats):
                    p_text = "n/a" if stats.get("monte_carlo_p") is None else f"{stats['monte_carlo_p']:.4f}"
                    return (
                        f"<tr><td>{name}</td><td>{stats['circuit_mean']:.4g}</td><td>{stats['random_mean']:.4g}</td>"
                        f"<td>{stats['ratio']:.2f}</td><td>{p_text}</td><td>{stats['scored_nodes']}</td></tr>"
                    )

                display(HTML(
                    "<table><tr><th>condition</th><th>circuit mean</th><th>random mean</th><th>enrichment</th><th>p</th><th>nodes</th></tr>"
                    + "".join(_h3_row(name, stats) for name, stats in v.get("conditions", {}).items())
                    + "</table>"
                ))
                ef = v.get("exercised_fraction")
                print(f"parents {v.get('parent_nodes')} (target excluded: {v.get('excluded_target_nodes')}); "
                      f"exercised {v.get('exercised_parents', v.get('measured_nodes'))} "
                      f"({'n/a' if ef is None else f'{ef:.0%}'}); mode {v.get('mode')}; draws "
                      f"{next(iter(v.get('conditions', {}).values()), {}).get('draws')}")
                sel = v.get("circuit_selectivity_target_over_placebo")
                if sel is not None:
                    print(f"circuit selectivity target/placebo: {sel:.2f}x")
                display(Markdown(f"**H3 verdict: {v.get('h3_verdict')}**"))
